# Analyse an Opta file with `wa-setpieces`

This notebook loads one Opta JSON export, validates it, runs the complete set-piece workflow, displays the results, and exports CSV tables plus an HTML report.

In [1]:
# Run once when using the repository checkout:
%pip install -e "..[viz,ml,convert]"

Obtaining file:///Users/marclamberts/Documents/GitHub/waltzinganalytics


  Installing build dependencies ... -

 \

 done


  Checking if build backend supports build_editable ... done


...

  Building editable for wa-setpieces (pyproject.toml) ... -

 done
  Created wheel for wa-setpieces: filename=wa_setpieces-0.30.1-0.editable-py3-none-any.whl size=15018 sha256=f5d3611d89ac7d9f74acc914abc81e7896e4cc3200302ca273cc1a31d23e4e16
  Stored in directory: /private/var/folders/zm/0c3h1wgs6y9914t7dqm_2nlw0000gn/T/pip-ephem-wheel-cache-ty92e9ap/wheels/65/ee/b3/d8a44a32ada8a0b031db0adb866a6074db3412e70167748066
Successfully built wa-setpieces


  Attempting uninstall: wa-setpieces
    Found existing installation: wa-setpieces 0.29.1
    Uninstalling wa-setpieces-0.29.1:


      Successfully uninstalled wa-setpieces-0.29.1


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

from wa_setpieces import (
    XTModel,
    defensive_rating,
    load_matches,
    run_workflow,
    validate_events,
    write_html_report,
)

## Configuration

Change `OPTA_FILE` to your Opta JSON export. For production added-value ratings, point `XT_MODEL_FILE` to an xT model trained on a season or larger sample.

In [3]:
OPTA_FILE = Path("../tests/data/sample_match.json")
SET_PIECE_TYPE = "corner"  # corner, free_kick, throw_in, goal_kick, kick_off, penalty
OUTPUT_DIR = Path("../analysis")
XT_MODEL_FILE = None  # Example: Path("../league_xt.npz")
FIT_XT_ON_THIS_MATCH = False  # Illustrative only; a single match is too small for production xT

## Load and validate the Opta events

In [4]:
events = validate_events(load_matches(OPTA_FILE))

print(f"Loaded {len(events):,} events")
print(f"Teams: {events['contestantId'].nunique()}")
events.head()

Loaded 1,613 events
Teams: 2


,matchId,id,eventId,typeId,periodId,timeMin,timeSec,contestantId,playerId,playerName,...,q_108,q_180,q_185,q_199,q_389,q_86,q_80,q_209,q_302,q_229
0,sample_match,2904150991,2,32,1,0,0,cxb4hqite921i36gwrezdts7c,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sample_match,2904150989,2,32,1,0,0,f2yd0yzt0om6qhks9gbowu1d6,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sample_match,2904150997,3,1,1,0,0,cxb4hqite921i36gwrezdts7c,arj6wz36r6wr961xynqg4m4lx,F. Cornejo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sample_match,2904151025,4,1,1,0,3,cxb4hqite921i36gwrezdts7c,d7d208zfvkuvjaba38rlcjxl,G. Valle,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sample_match,2904151045,3,2,1,0,8,f2yd0yzt0om6qhks9gbowu1d6,9g05j8p6gyul0zlnx70rffkve,S. Mina,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load or fit the xT model

In [5]:
model = None
if XT_MODEL_FILE is not None:
    model = XTModel.load(XT_MODEL_FILE)
    print("Loaded xT model:", model.metadata)
elif FIT_XT_ON_THIS_MATCH:
    model = XTModel.fit(events)
    print("Warning: xT was fitted on one match and is illustrative only.")
else:
    print("Running without xT. Added-value and player-rating tables will be unavailable.")

Running without xT. Added-value and player-rating tables will be unavailable.


## Run the complete workflow

In [6]:
result = run_workflow(events, SET_PIECE_TYPE, model=model)
result.summary

,contestantId,set_piece_type,attempts,successful,success_rate,shots,goals
0,cxb4hqite921i36gwrezdts7c,corner,2,1,0.500,0,0
1,f2yd0yzt0om6qhks9gbowu1d6,corner,7,1,0.143,0,0


## Inspect the available tables

In [7]:
tables = {
    name: value
    for name, value in vars(result).items()
    if isinstance(value, pd.DataFrame)
}

if not result.defensive_summary.empty:
    tables["defensive_rating"] = defensive_rating(result.defensive_summary)

pd.DataFrame(
    [{"table": name, "rows": len(table), "columns": len(table.columns)}
     for name, table in tables.items()]
)

,table,rows,columns
0,summary,2,7
1,team_counts,2,5
2,player_counts,6,7
3,deliveries,9,10
4,second_phases,9,15
5,retention,9,6
6,report,2,8
7,team_rating,2,11
8,defensive_summary,2,9
9,defensive_routine_summary,3,6


In [8]:
# Examples of individual outputs:
display(result.report)
display(result.defensive_summary)
display(result.routine_summary)
display(result.routine_team_profiles)
display(result.routine_taker_profiles)
display(result.routine_target_matrix)
display(result.routines.head())
display(result.first_contacts.head())

,contestantId,attempts,successful,success_rate,second_phases,second_phase_goals,second_phase_rate,retention_rate
0,cxb4hqite921i36gwrezdts7c,2,1,0.500,0,0,0.000,0.500
1,f2yd0yzt0om6qhks9gbowu1d6,7,1,0.143,2,0,0.286,0.429


,contestantId,set_piece_type,matches,attempts_faced,opponent_successful,shots_conceded,goals_conceded,opponent_success_rate,shots_conceded_per_100
0,cxb4hqite921i36gwrezdts7c,corner,1,7,1,0,0,0.143,0.0
1,f2yd0yzt0om6qhks9gbowu1d6,corner,1,2,1,0,0,0.500,0.0


,contestantId,routine_type,attempts,successful,retained,shots,goals,avg_distance,avg_progression,usage_share,success_rate,retention_rate,shot_rate
0,cxb4hqite921i36gwrezdts7c,penalty_area,2,1,1,0,0,44.81,-9.00,1.000,0.5,0.500,0.0
1,f2yd0yzt0om6qhks9gbowu1d6,central_six_yard,6,0,2,0,0,45.15,-4.63,0.857,0.0,0.333,0.0
2,f2yd0yzt0om6qhks9gbowu1d6,short,1,1,1,0,0,10.03,-4.00,0.143,1.0,1.000,0.0


,contestantId,set_piece_type,attempts,routine_families,distinct_patterns,most_used_routine,most_used_pattern,top_pattern_share,routine_diversity,success_rate,retention_rate,shot_rate,goal_rate,avg_distance_m,avg_progression
0,cxb4hqite921i36gwrezdts7c,corner,2,1,1,penalty_area,penalty_area|right|penalty_area,1.000,0.000,0.500,0.500,0.0,0.0,31.31,-9.00
1,f2yd0yzt0om6qhks9gbowu1d6,corner,7,2,4,central_six_yard,central_six_yard|right|six_yard_box,0.429,0.921,0.143,0.429,0.0,0.0,27.60,-4.54


,playerId,playerName,contestantId,set_piece_type,attempts,preferred_routine,preferred_routine_share,routine_families,success_rate,retention_rate,shots_created,goals_created,avg_distance_m,avg_progression
0,c74dqd8wbe059xsmdo2twmec4,S. Vasquez,f2yd0yzt0om6qhks9gbowu1d6,corner,3,central_six_yard,0.667,2,0.333,1.0,0,0,20.8,-3.6


,contestantId,routine_type,destination_zone,attempts,successful,shots,goals,team_usage_share,success_rate,shot_rate
0,cxb4hqite921i36gwrezdts7c,penalty_area,penalty_area,2,1,0,0,1.000,0.5,0.0
1,f2yd0yzt0om6qhks9gbowu1d6,central_six_yard,penalty_area,1,0,0,0,0.143,0.0,0.0
2,f2yd0yzt0om6qhks9gbowu1d6,central_six_yard,six_yard_box,5,0,0,0,0.714,0.0,0.0
3,f2yd0yzt0om6qhks9gbowu1d6,short,wide_final_third,1,1,0,0,0.143,1.0,0.0


,matchId,eventId,contestantId,playerId,playerName,set_piece_type,routine_type,delivery_technique,post_target,x,...,target_channel,destination_zone,successful,retained,shots,goals,delivery_outcome,routine_key,timeMin,timeSec
0,sample_match,191,f2yd0yzt0om6qhks9gbowu1d6,d6gphu89283pngbaenacu1cr9,R. Jaramillo,corner,central_six_yard,inswinger,near_post,99.9,...,central,six_yard_box,False,False,0,0,lost,central_six_yard|left|six_yard_box,27,33
1,sample_match,223,cxb4hqite921i36gwrezdts7c,cebzrizhi195acmaw8c4cku09,A. Tobar,corner,penalty_area,None,near_post,99.5,...,right_half_space,penalty_area,True,True,0,0,retained,penalty_area|right|penalty_area,32,16
2,sample_match,404,cxb4hqite921i36gwrezdts7c,cebzrizhi195acmaw8c4cku09,A. Tobar,corner,penalty_area,inswinger,central,99.5,...,central,penalty_area,False,False,0,0,lost,penalty_area|right|penalty_area,49,20
3,sample_match,482,f2yd0yzt0om6qhks9gbowu1d6,c9htznfxd2gjo5s3klq2tvlqt,D. Armas,corner,central_six_yard,outswinger,central,99.5,...,central,penalty_area,False,False,0,0,lost,central_six_yard|left|penalty_area,55,6
4,sample_match,610,f2yd0yzt0om6qhks9gbowu1d6,c74dqd8wbe059xsmdo2twmec4,S. Vasquez,corner,central_six_yard,inswinger,near_post,99.4,...,central,six_yard_box,False,True,0,0,retained,central_six_yard|right|six_yard_box,68,7


,matchId,eventId,contestantId,takerId,takerName,first_contact_player_id,first_contact_player_name,first_contact_team_id,first_contact_won,seconds_to_contact,confidence
0,sample_match,191,f2yd0yzt0om6qhks9gbowu1d6,d6gphu89283pngbaenacu1cr9,R. Jaramillo,arj6wz36r6wr961xynqg4m4lx,F. Cornejo,cxb4hqite921i36gwrezdts7c,False,3.0,event_sequence
1,sample_match,223,cxb4hqite921i36gwrezdts7c,cebzrizhi195acmaw8c4cku09,A. Tobar,9cjt8bpelefv3ht1kgcw0hlas,J. Caicedo,cxb4hqite921i36gwrezdts7c,True,3.0,event_sequence
2,sample_match,404,cxb4hqite921i36gwrezdts7c,cebzrizhi195acmaw8c4cku09,A. Tobar,9g05j8p6gyul0zlnx70rffkve,S. Mina,f2yd0yzt0om6qhks9gbowu1d6,False,3.0,event_sequence
3,sample_match,482,f2yd0yzt0om6qhks9gbowu1d6,c9htznfxd2gjo5s3klq2tvlqt,D. Armas,ecaofku4htx8if4kk2teq1rrp,R. Adé,cxb4hqite921i36gwrezdts7c,False,3.0,event_sequence
4,sample_match,610,f2yd0yzt0om6qhks9gbowu1d6,c74dqd8wbe059xsmdo2twmec4,S. Vasquez,eu1htvf3chz0kcrl5paflxzv9,Deyverson,cxb4hqite921i36gwrezdts7c,False,3.0,event_sequence


## Export CSV tables and an HTML report

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, table in tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

report_path = write_html_report(
    OUTPUT_DIR / "report.html",
    title=f"{SET_PIECE_TYPE.replace('_', ' ').title()} analysis",
    tables=tables,
    methodology=(
        "Source: Opta event data. Retention, phases, first contact and added value "
        "are derived event-data heuristics. Ratings need a season-sized benchmark."
    ),
)

print(f"Wrote {len(tables)} CSV files")
print(f"HTML report: {report_path.resolve()}")

Wrote 21 CSV files
HTML report: /Users/marclamberts/Documents/GitHub/waltzinganalytics/analysis/report.html
